In [11]:
import sys
sys.path.insert(0, '../Week-5-6-7-8')

import json
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, NuSVR

from helpers.modeling import identify_column_types, create_preprocessor
from uncertainty import run_conformal, calibration_table, prepare_data_group

with open('../Week-5-6-7-8/results.json', 'r') as f:
    results = json.load(f)

df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")

In [12]:
target_col = 'cs_28d'

# Group-aware split: no publication bleeds across train / val / test
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data_group(
    df, target_col, group_col='paper_reference'
)

pub_train = df.loc[X_train.index, 'paper_reference']
pub_test = df.loc[X_test.index, 'paper_reference']
pub_val  = df.loc[X_val.index,  'paper_reference']

print(f"Train size : {X_train.shape[0]}  ({pub_train.nunique()} publications)")
print(f"Val size   : {X_val.shape[0]}  ({pub_val.nunique()} publications)")
print(f"Test size  : {X_test.shape[0]}  ({pub_test.nunique()} publications)")

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X_train)
preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns,
                                   handle_unknown='ignore')
print(f"\nEncoded training shape: {preprocessor.fit_transform(X_train, y_train).shape}")

Train size : 1461  (115 publications)
Val size   : 329  (25 publications)
Test size  : 283  (25 publications)

Encoded training shape: (1461, 69)


In [13]:
pipeline_knn, train_knn, test_knn, q_knn, intervals_knn, calibration_knn = run_conformal(
    KNeighborsRegressor, 'knn', results, 'recoded_50', preprocessor,
    X_train, y_train, X_val, y_val, X_test, y_test, pub_test
)
print(f"Train: {train_knn}")
print(f"Test:  {test_knn}")
print(f"Conformal q (90%): {q_knn:.2f}")
calibration_knn

Train: {'RMSE': np.float64(6.549791070260361), 'MAE': 3.885417120117953, 'R2': 0.9601099924523562, 'Correlation': np.float64(0.9799886467693996), 'Mean_Residual': np.float64(-0.0694675872902154), 'N': 1461}
Test:  {'RMSE': np.float64(22.71878601115441), 'MAE': 17.143023793679394, 'R2': 0.42745663052453675, 'Correlation': np.float64(0.7359099764715195), 'Mean_Residual': np.float64(-6.958370812021887), 'N': 283}
Conformal q (90%): 74.31


,publication,n_rows,mean_true,mean_prediction,coverage,interval_width,mean_residual,mean_distance
0,Ref-107-Research,16,109.031250,111.424252,1.0,148.623231,7.428191,15.121621
1,Ref-11-Research,4,127.750000,115.287702,1.0,148.623231,12.462298,9.939555
2,Ref-113-Research,5,165.055660,170.200181,1.0,148.623231,8.389570,5.889168
3,Ref-114-Research,5,208.804000,191.939738,1.0,148.623231,16.864262,6.097123
4,Ref-116-Research,36,152.627778,154.624389,1.0,148.623231,4.013374,4.158053
5,Ref-117-Research,5,113.000000,113.692291,1.0,148.623231,11.059342,17.842439
6,Ref-121-Research,80,165.075000,182.913925,1.0,148.623231,22.948058,8.820403
7,Ref-126-Research,5,114.600000,142.689382,1.0,148.623231,28.089382,5.621627
8,Ref-127-Research,3,201.200000,153.465759,1.0,148.623231,47.734241,4.881362
9,Ref-128-Research,3,201.266667,200.345212,1.0,148.623231,2.165204,3.440041


In [14]:
pipeline_svr, train_svr, test_svr, q_svr, intervals_svr, calibration_svr = run_conformal(
    SVR, 'svr', results, 'recoded_50', preprocessor,
    X_train, y_train, X_val, y_val, X_test, y_test, pub_test
)
print(f"Train: {train_svr}")
print(f"Test:  {test_svr}")
print(f"Conformal q (90%): {q_svr:.2f}")
calibration_svr

Train: {'RMSE': np.float64(8.40126124316765), 'MAE': 5.4071940633825255, 'R2': 0.9343706347141592, 'Correlation': np.float64(0.9667990257844075), 'Mean_Residual': np.float64(-0.06775276318929017), 'N': 1461}
Test:  {'RMSE': np.float64(26.499205995130957), 'MAE': 20.59534876275175, 'R2': 0.221060244139054, 'Correlation': np.float64(0.635116646889457), 'Mean_Residual': np.float64(-3.9547328885356996), 'N': 283}
Conformal q (90%): 83.52


,publication,n_rows,mean_true,mean_prediction,coverage,interval_width,mean_residual,mean_distance
0,Ref-107-Research,16,109.031250,122.880283,1.0,167.049789,19.392585,6.604180
1,Ref-11-Research,4,127.750000,124.901602,1.0,167.049789,4.751940,3.658512
2,Ref-113-Research,5,165.055660,168.759424,1.0,167.049789,3.703764,3.133568
3,Ref-114-Research,5,208.804000,181.229420,1.0,167.049789,27.574580,2.200367
4,Ref-116-Research,36,152.627778,140.991987,1.0,167.049789,11.635790,1.931499
5,Ref-117-Research,5,113.000000,128.451172,1.0,167.049789,15.451172,8.214006
6,Ref-121-Research,80,165.075000,169.439001,1.0,167.049789,15.123495,3.887524
7,Ref-126-Research,5,114.600000,159.117366,1.0,167.049789,44.517366,2.476770
8,Ref-127-Research,3,201.200000,150.940782,1.0,167.049789,50.259218,2.086621
9,Ref-128-Research,3,201.266667,210.395746,1.0,167.049789,9.129080,2.164416


In [15]:
pipeline_nusvr, train_nusvr, test_nusvr, q_nusvr, intervals_nusvr, calibration_nusvr = run_conformal(
    NuSVR, 'nusvr', results, 'recoded_50', preprocessor,
    X_train, y_train, X_val, y_val, X_test, y_test, pub_test
)
print(f"Train: {train_nusvr}")
print(f"Test:  {test_nusvr}")
print(f"Conformal q (90%): {q_nusvr:.2f}")
calibration_nusvr

Train: {'RMSE': np.float64(8.345009428865133), 'MAE': 5.556364844356813, 'R2': 0.9352465535504952, 'Correlation': np.float64(0.9672652519299139), 'Mean_Residual': np.float64(-0.01251630557359761), 'N': 1461}
Test:  {'RMSE': np.float64(27.34405172102479), 'MAE': 21.205695905824264, 'R2': 0.17060028582318842, 'Correlation': np.float64(0.6113311468749365), 'Mean_Residual': np.float64(-3.043244004587312), 'N': 283}
Conformal q (90%): 86.04


,publication,n_rows,mean_true,mean_prediction,coverage,interval_width,mean_residual,mean_distance
0,Ref-107-Research,16,109.031250,123.098967,1.0,172.081823,19.272839,6.603872
1,Ref-11-Research,4,127.750000,124.549005,1.0,172.081823,6.551198,3.657237
2,Ref-113-Research,5,165.055660,165.723739,1.0,172.081823,1.726145,3.133449
3,Ref-114-Research,5,208.804000,183.378428,1.0,172.081823,26.267590,2.199386
4,Ref-116-Research,36,152.627778,138.754490,1.0,172.081823,13.873288,1.931494
5,Ref-117-Research,5,113.000000,126.857361,1.0,172.081823,13.857361,8.213561
6,Ref-121-Research,80,165.075000,168.755458,1.0,172.081823,15.317576,3.886623
7,Ref-126-Research,5,114.600000,170.166651,1.0,172.081823,55.566651,2.476634
8,Ref-127-Research,3,201.200000,151.581441,1.0,172.081823,49.618559,2.086791
9,Ref-128-Research,3,201.266667,209.518416,1.0,172.081823,8.251749,2.164416


In [18]:
poor_coverage_threshold = 1

poor_coverage = pd.concat([
    calibration_knn.assign(model='knn'),
    calibration_svr.assign(model='svr'),
    calibration_nusvr.assign(model='nusvr'),
])
poor_coverage = (
    poor_coverage[poor_coverage['coverage'] < poor_coverage_threshold]
    .sort_values('coverage')
    [['model', 'publication', 'n_rows', 'coverage', 'interval_width', 'mean_residual', 'mean_distance']]
    .reset_index(drop=True)
)

print(f"Publications below {poor_coverage_threshold:.0%} coverage: {poor_coverage['publication'].nunique()} unique, {len(poor_coverage)} model-publication pairs")
poor_coverage

Publications below 100% coverage: 1 unique, 1 model-publication pairs


,model,publication,n_rows,coverage,interval_width,mean_residual,mean_distance
0,svr,Ref-93-Research,10,0.9,167.049789,29.792381,3.072411


In [ ]:
import matplotlib.pyplot as plt

all_cal = pd.concat([
    calibration_knn.assign(model='knn'),
    calibration_svr.assign(model='svr'),
    calibration_nusvr.assign(model='nusvr'),
], ignore_index=True)

colors = {'knn': 'steelblue', 'svr': 'tomato', 'nusvr': 'seagreen'}

fig, ax = plt.subplots(figsize=(9, 6))

for model_name, grp in all_cal.groupby('model'):
    ax.scatter(grp['mean_distance'], grp['mean_residual'],
               label=model_name.upper(), color=colors[model_name],
               alpha=0.7, edgecolors='white', linewidths=0.4, s=60)

# label outliers: top 5 by mean_residual per model
for model_name, grp in all_cal.groupby('model'):
    for _, row in grp.nlargest(5, 'mean_residual').iterrows():
        ax.annotate(row['publication'].replace('-Research', '').replace('-data', ''),
                    xy=(row['mean_distance'], row['mean_residual']),
                    fontsize=7, color=colors[model_name],
                    xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean Distance to Training (5-NN)', fontsize=11)
ax.set_ylabel('Mean Absolute Residual (MPa)', fontsize=11)
ax.set_title('Distance vs Residual per Publication — Group Split', fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [20]:
print(q_knn, q_svr, q_nusvr)

74.31161544924942 83.52489458133604 86.04091140249217
